# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the **FAIR^2** dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is described by a [Croissant schema](https://mlcommons.org/croissant/) at the following URL:

```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```

In [ ]:
# Ensure `mlcroissant` is installed (uncomment if needed in new environments)
!pip install mlcroissant pandas

## 1. Data Loading
Load the metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define Croissant metadata URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)

print(f"Dataset loaded: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")
print(f"Dataset Identifier: {dataset.metadata.identifier}")
print(f"Version: {dataset.metadata.version}")

## 2. Data Overview
Review available record sets, fields, and their unique `@id`s. The `@id` allows us to refer precisely to specific data elements in the Croissant schema.

In [ ]:
# List available record sets and their @id's
print("Record sets available in this dataset:")
if hasattr(dataset.metadata, 'record_sets'):
    for rs in dataset.metadata.record_sets:
        print(f"- {rs['@id']} (name: {rs['name']})")
else:
    # Try fallback: search the metadata dict or Croissant-compatible object
    rs_list = getattr(dataset.metadata, 'recordSet', [])
    if not rs_list:
        # Try for datasets with no recordSet, but that get field definitions directly
        print("No record sets found in metadata. Trying to list possible fields or columns...")
    for rs in rs_list:
        print(f"- {rs['@id']} (name: {rs.get('name', '')})")

# For this dataset, let's programmatically get all record set IDs from the loaded object
if hasattr(dataset, 'record_sets'):
    available_record_set_ids = list(dataset.record_sets.keys())
else:
    # fallback for datasets not conforming exactly to mlcroissant API
    available_record_set_ids = []

print("\nAvailable record set @id's:")
for rs_id in available_record_set_ids:
    print(f"  - {rs_id}")

# For demonstration, print field @id for first available record set
if available_record_set_ids:
    first_rs = available_record_set_ids[0]
    print(f"\nFields in record set {first_rs}:")
    try:
        fields = dataset.record_sets[first_rs]['fields']
        for f in fields:
            print(f"  - {f['@id']} ({f['name']})")
    except Exception as e:
        print("  Could not list fields for this record set:", e)
else:
    print("No record sets found in dataset.")

## 3. Data Extraction
Extract records for each record set into DataFrames for further analysis.

All entities such as record sets and fields are referenced by their unique `@id` as per Croissant specification.

In [ ]:
# Extract data from each record set referenced by their @id
dataframes = {}
record_set_ids = list(dataset.record_sets.keys()) # List of @ids for all record sets
print(f"Extracting data for record set ids: {record_set_ids}")

for record_set_id in record_set_ids:
    print(f"\nLoading records for record set: {record_set_id}")
    # Each record is a dict with field @id as keys
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  Loaded {len(df)} records.")
        print(f"  Columns (@id): {list(df.columns)}\n")
        # Show a sample
        display(df.head())
    else:
        print("  This record set contains no records.")

## 4. Exploratory Data Analysis (EDA)
Below are examples of filtering, normalization, and grouping by key categorical columns.

Replace `<numeric_field_id>` and `<group_field_id>` with actual `@id` values for numeric and grouping fields in your dataset.

In [ ]:
# Example: Filter and normalize numeric field, group by a categorical field
from IPython.display import display

# Choose a record set with tabular data. Here we use the first one as example.
if record_set_ids:
    rs_id = record_set_ids[0]
    df = dataframes[rs_id]
    print(f"Using record set: {rs_id}")
    print(f"Available columns (@id): {df.columns.tolist()}")
    # Try to select a numeric field by inspecting the columns
    numeric_field_id = None
    for col in df.columns:
        # Try to heuristically find a likely numeric field
        if any(key in col.lower() for key in ["age", "interval", "number", "count", "size"]):
            numeric_field_id = col
            break
    if numeric_field_id is None and len(df.columns)>0:
        # fallback to the first column
        numeric_field_id = df.columns[0]
    print(f"Selected numeric field: {numeric_field_id}")

    # Attempt to cast to float if not already numeric
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

    # Filter: e.g., field values > threshold (change threshold as appropriate)
    threshold = df[numeric_field_id].quantile(0.2)  # use 20th percentile as threshold example
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize the numeric field (mean/std scaling)
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try categorizing/grouping by a likely grouping field
    group_field_id = None
    for col in df.columns:
        if any(key in col.lower() for key in ["sex", "gender", "location", "anatomy", "site", "group", "status", "type"]):
            group_field_id = col
            break
    print(f"Grouping by: {group_field_id}")
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"Grouped data: mean {numeric_field_id} by {group_field_id}")
        display(grouped_df)
else:
    print("No record sets present or data unavailable for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. We'll use basic matplotlib and seaborn for plotting numeric fields and their distributions by categories if present.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_ids and numeric_field_id:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
    
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()

## 6. Conclusion
This notebook demonstrated how to:
- Load a Croissant-structured dataset with `mlcroissant`
- Explore record sets and fields using their `@id`
- Extract records to pandas DataFrames
- Perform basic exploratory processing and normalization
- Visualize field distributions and group comparisons

Further analytics can leverage the clean DataFrames and the precise referencing of all entities by `@id` for robust programmatic processing.